# Agent 1 — Module 3 Final Updated Notebook

## Purpose

This notebook runs the **current production Module 3 code** from the `Agent_1` project.

It deliberately does **not** copy Module 3 classes into notebook cells. Copying the code creates a second, outdated version whenever the `.py` files are improved.

The notebook therefore uses the same modules and scripts as these commands:

```powershell
python -m scripts.test_module_3_regressions
python -m scripts.test_module_3_batch
```

This means the notebook output and command-line output come from the same source of truth.

## 1. Project setup

Place this notebook anywhere inside the `Agent_1` project, preferably in the project root, and select the project's `.venv` kernel in Jupyter or VS Code.

In [ ]:
from __future__ import annotations

import csv
import json
import os
import subprocess
import sys
from pathlib import Path
from typing import Iterable

import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    candidates = [start, *start.parents]

    for parent in list(candidates):
        candidates.append(parent / "Agent_1")

    seen: set[Path] = set()

    for candidate in candidates:
        candidate = candidate.resolve()

        if candidate in seen:
            continue

        seen.add(candidate)

        if (
            (candidate / "app").is_dir()
            and (candidate / "scripts").is_dir()
            and (candidate / "requirements.txt").is_file()
        ):
            return candidate

    raise RuntimeError(
        "Agent_1 project root was not found. "
        "Move this notebook inside the Agent_1 project and run this cell again."
    )


try:
    notebook_start = Path.cwd()
except Exception:
    notebook_start = Path(".")

PROJECT_ROOT = find_project_root(notebook_start)
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root : {PROJECT_ROOT}")
print(f"Python       : {sys.executable}")
print(f"Python ver.  : {sys.version.split()[0]}")
print(f"Working dir  : {Path.cwd()}")

## 2. Verify the active environment

The Python path should normally point to:

```text
Agent_1/.venv/Scripts/python.exe
```

If it does not, select the `.venv` notebook kernel before continuing.

In [ ]:
expected_venv = PROJECT_ROOT / ".venv"

active_python = Path(sys.executable).resolve()
using_project_venv = expected_venv.resolve() in active_python.parents

print(f"Expected venv: {expected_venv}")
print(f"Active Python: {active_python}")
print(f"Using project .venv: {using_project_venv}")

if not using_project_venv:
    print(
        "\nWARNING: The notebook is not using Agent_1/.venv. "
        "The code may still run, but selecting the project kernel is recommended."
    )

## 3. Helper for exact command execution

This helper runs Python modules with the notebook's active Python interpreter and prints their output as it is produced.

In [ ]:
def run_module(
    module: str,
    arguments: Iterable[str] = (),
    *,
    stop_on_error: bool = True,
) -> subprocess.CompletedProcess[str]:
    command = [
        sys.executable,
        "-m",
        module,
        *list(arguments),
    ]

    print("\n" + "=" * 110)
    print("RUNNING:")
    print(" ".join(f'"{part}"' if " " in part else part for part in command))
    print("=" * 110 + "\n")

    process = subprocess.Popen(
        command,
        cwd=PROJECT_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )

    output_lines: list[str] = []

    assert process.stdout is not None

    for line in process.stdout:
        print(line, end="")
        output_lines.append(line)

    return_code = process.wait()
    combined_output = "".join(output_lines)

    result = subprocess.CompletedProcess(
        args=command,
        returncode=return_code,
        stdout=combined_output,
        stderr=None,
    )

    print("\n" + "-" * 110)
    print(f"Exit code: {return_code}")
    print("-" * 110)

    if stop_on_error and return_code != 0:
        raise RuntimeError(
            f"Command failed with exit code {return_code}: "
            f"python -m {module}"
        )

    return result

# Part A — Verify today's Module 3 improvements

The regression suite checks the current production implementation, including:

- false-positive protection;
- official AQA mappings;
- evidence competition;
- continuation handling;
- unmapped-topic detection;
- recap-evidence protection;
- primary/supporting ranking.

In [ ]:
regression_result = run_module(
    "scripts.test_module_3_regressions"
)

## Regression status

In [ ]:
if regression_result.returncode == 0:
    print("ALL MODULE 3 REGRESSION TESTS PASSED")
else:
    print("MODULE 3 REGRESSION TESTS FAILED")

# Part B — Run the exact Module 3 batch script

The next cell is equivalent to:

```powershell
python -m scripts.test_module_3_batch
```

It uses the script's own default input and output directories, so its terminal output is the same production output you receive from PowerShell.

In [ ]:
batch_result = run_module(
    "scripts.test_module_3_batch"
)

## Optional batch controls

Do not run this cell unless you want custom arguments.

Available arguments from the current script are:

```text
--input-dir
--output-dir
--limit
```

`--no-llm` is not supported by this batch script.

In [ ]:
# Example: run only the first two transcripts.
#
# limited_batch_result = run_module(
#     "scripts.test_module_3_batch",
#     ["--limit", "2"],
# )
#
# Example: specify an input directory.
#
# custom_batch_result = run_module(
#     "scripts.test_module_3_batch",
#     [
#         "--input-dir",
#         "test_outputs/module_1_2_batch",
#         "--output-dir",
#         "test_outputs/module_3_batch",
#     ],
# )

# Part C — Find and display the batch output

The batch script remains responsible for generating the official files. The notebook only finds and displays those files; it does not recalculate or alter Module 3 results.

In [ ]:
def find_recent_files(
    root: Path,
    filename_patterns: tuple[str, ...],
) -> list[Path]:
    matches: list[Path] = []

    if not root.exists():
        return matches

    for pattern in filename_patterns:
        matches.extend(root.rglob(pattern))

    unique_matches = {
        path.resolve()
        for path in matches
        if path.is_file()
    }

    return sorted(
        unique_matches,
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )


TEST_OUTPUTS = PROJECT_ROOT / "test_outputs"

summary_files = find_recent_files(
    TEST_OUTPUTS,
    (
        "*batch*summary*.csv",
        "*batch*summary*.json",
        "batch_summary.csv",
        "batch_summary.json",
    ),
)

print("Most recent batch summary files:")

if not summary_files:
    print("No summary files were found under test_outputs.")
else:
    for index, path in enumerate(summary_files[:10], start=1):
        modified = path.stat().st_mtime
        print(
            f"{index:>2}. "
            f"{path.relative_to(PROJECT_ROOT)}"
        )

## Display the most recent CSV summary

In [ ]:
csv_summaries = [
    path
    for path in summary_files
    if path.suffix.lower() == ".csv"
]

latest_csv_summary: Path | None = (
    csv_summaries[0]
    if csv_summaries
    else None
)

if latest_csv_summary is None:
    print("No batch summary CSV was found.")
else:
    print(
        "Displaying:",
        latest_csv_summary.relative_to(PROJECT_ROOT),
    )

    batch_summary_df = pd.read_csv(
        latest_csv_summary
    )

    display(batch_summary_df)

## Display the most recent JSON summary

In [ ]:
json_summaries = [
    path
    for path in summary_files
    if path.suffix.lower() == ".json"
    and "error" not in path.name.lower()
]

latest_json_summary: Path | None = (
    json_summaries[0]
    if json_summaries
    else None
)

if latest_json_summary is None:
    print("No batch summary JSON was found.")
else:
    print(
        "Displaying:",
        latest_json_summary.relative_to(PROJECT_ROOT),
    )

    with latest_json_summary.open(
        "r",
        encoding="utf-8",
    ) as file:
        batch_summary_json = json.load(file)

    if isinstance(batch_summary_json, list):
        display(
            pd.DataFrame(batch_summary_json)
        )
    else:
        print(
            json.dumps(
                batch_summary_json,
                indent=2,
                ensure_ascii=False,
            )
        )

# Part D — Inspect generated topic outputs

The following cell lists the most recently generated readable topic files and JSON topic files.

In [ ]:
topic_output_files = find_recent_files(
    TEST_OUTPUTS,
    (
        "topics_readable.txt",
        "merged_topics_readable.txt",
        "topics.json",
        "merged_topics.json",
    ),
)

if not topic_output_files:
    print("No topic output files were found under test_outputs.")
else:
    print("Most recent topic output files:")

    for index, path in enumerate(
        topic_output_files[:20],
        start=1,
    ):
        print(
            f"{index:>2}. "
            f"{path.relative_to(PROJECT_ROOT)}"
        )

## Display one readable topic result

Change `RESULT_INDEX` to inspect another file from the list above.

In [ ]:
RESULT_INDEX = 0

readable_outputs = [
    path
    for path in topic_output_files
    if path.suffix.lower() == ".txt"
]

if not readable_outputs:
    print("No readable topic output was found.")
elif RESULT_INDEX >= len(readable_outputs):
    print(
        f"RESULT_INDEX must be between 0 and "
        f"{len(readable_outputs) - 1}."
    )
else:
    selected_result = readable_outputs[
        RESULT_INDEX
    ]

    print(
        "Displaying:",
        selected_result.relative_to(PROJECT_ROOT),
    )
    print("=" * 110)
    print(
        selected_result.read_text(
            encoding="utf-8",
            errors="replace",
        )
    )

# Part E — Run one complete transcript through Agent 1

This optional cell uses the current production pipeline.

Set `TRANSCRIPT_FILE` to the real file path and run the cell.

In [ ]:
TRANSCRIPT_FILE = (
    PROJECT_ROOT
    / "test_data"
    / "Transcript_Test_1_Data_Representation.docx"
)

if not TRANSCRIPT_FILE.exists():
    print(
        "Transcript not found:",
        TRANSCRIPT_FILE,
    )
    print(
        "Update TRANSCRIPT_FILE with the correct path "
        "before running this section."
    )
else:
    single_transcript_result = run_module(
        "scripts.run_agent1_pipeline",
        [
            "--file",
            str(TRANSCRIPT_FILE),
            "--no-llm",
        ],
    )

# Final architecture represented by this notebook

```text
Current production Module 1 and Module 2 outputs
                ↓
Current production Module 3 pipeline
                ↓
Candidate extraction
                ↓
CS relevance filtering
                ↓
Evidence-quality evaluation
                ↓
Continuation handling
                ↓
Unmapped-CS detection
                ↓
Evidence competition
                ↓
Topic merging
                ↓
Primary / Supporting ranking
                ↓
Existing batch JSON, text and CSV outputs
```

## Why this notebook remains correct after later code changes

The notebook imports and executes the `.py` implementation. Therefore:

- there is no duplicate Module 3 implementation;
- today's fixes are automatically used;
- future Qdrant integration will also be used once added to the production pipeline;
- CLI and notebook output remain aligned;
- regression tests verify the implementation before batch execution.